# Домашняя работа 5. Polars, ускорение pandas и оптимизация типов

## 1. Чтение датасета train.csv с помощью polars

In [1]:
"""Задача 1. Чтение датасета «Титаник» библиотекой polars.

Оба файла — train.csv и Housing.csv — лежат или рядом с ноутбуком (так бывает в
Colab), или в папке DataSet/ (так они лежат в репозитории), поэтому путь ищется,
а не вписывается жёстко.
"""

from pathlib import Path

import polars as pl


def find_dataset(filename: str) -> Path:
    """Ищет файл датасета в текущей папке, в DataSet/ и выше по дереву каталогов."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (folder / filename, folder / "DataSet" / filename):
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f"не найден файл {filename}")


TITANIC = find_dataset("train.csv")
HOUSING = find_dataset("Housing.csv")

# В polars нет отдельного индекса, как в pandas: PassengerId остаётся столбцом
titanic = pl.read_csv(TITANIC)
print(f"строк: {titanic.height}, столбцов: {titanic.width}")
titanic.head()

строк: 891, столбцов: 12


PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S"""
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S"""


## 2. Основная информация о датасете

In [2]:
# Типы данных: polars определяет их сам, schema возвращает пары «столбец — тип»
print("--- типы данных (dtypes) ---")
for column, dtype in titanic.schema.items():
    print(f"{column:<12} {dtype}")

# null_count() считает пропуски в каждом столбце; transpose() разворачивает
# строку в таблицу, чтобы её было удобно читать
nulls = (
    titanic.null_count()
    .transpose(include_header=True, header_name="столбец", column_names=["пропусков"])
    .with_columns((pl.col("пропусков") / titanic.height * 100).round(2).alias("процент"))
)
print("\n--- пропуски (null_count) ---")
print(nulls)

# Средние значения: pl.col(pl.Int64, pl.Float64) выбирает числовые столбцы по типу
print("\n--- средние значения (mean) ---")
print(
    titanic.select(pl.col(pl.Int64, pl.Float64).mean())
    .transpose(include_header=True, header_name="столбец", column_names=["среднее"])
)

# describe() — общая сводка: count, null_count, mean, std, min, квартили, max
titanic.describe()

--- типы данных (dtypes) ---
PassengerId  Int64
Survived     Int64
Pclass       Int64
Name         String
Sex          String
Age          Float64
SibSp        Int64
Parch        Int64
Ticket       String
Fare         Float64
Cabin        String
Embarked     String

--- пропуски (null_count) ---
shape: (12, 3)
┌─────────────┬───────────┬─────────┐
│ столбец     ┆ пропусков ┆ процент │
│ ---         ┆ ---       ┆ ---     │
│ str         ┆ u64       ┆ f64     │
╞═════════════╪═══════════╪═════════╡
│ PassengerId ┆ 0         ┆ 0.0     │
│ Survived    ┆ 0         ┆ 0.0     │
│ Pclass      ┆ 0         ┆ 0.0     │
│ Name        ┆ 0         ┆ 0.0     │
│ Sex         ┆ 0         ┆ 0.0     │
│ …           ┆ …         ┆ …       │
│ Parch       ┆ 0         ┆ 0.0     │
│ Ticket      ┆ 0         ┆ 0.0     │
│ Fare        ┆ 0         ┆ 0.0     │
│ Cabin       ┆ 687       ┆ 77.1    │
│ Embarked    ┆ 2         ┆ 0.22    │
└─────────────┴───────────┴─────────┘

--- средние значения (mean) ---
shape: (7

statistic,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
str,f64,f64,f64,str,str,f64,f64,f64,str,f64,str,str
"""count""",891.0,891.0,891.0,"""891""","""891""",714.0,891.0,891.0,"""891""",891.0,"""204""","""889"""
"""null_count""",0.0,0.0,0.0,"""0""","""0""",177.0,0.0,0.0,"""0""",0.0,"""687""","""2"""
"""mean""",446.0,0.383838,2.308642,null,null,29.699118,0.523008,0.381594,null,32.204208,null,null
"""std""",257.353842,0.486592,0.836071,null,null,14.526497,1.102743,0.806057,null,49.693429,null,null
"""min""",1.0,0.0,1.0,"""Abbing, Mr. Anthony""","""female""",0.42,0.0,0.0,"""110152""",0.0,"""A10""","""C"""
"""25%""",224.0,0.0,2.0,null,null,20.0,0.0,0.0,null,7.925,null,null
"""50%""",446.0,0.0,3.0,null,null,28.0,0.0,0.0,null,14.4542,null,null
"""75%""",669.0,1.0,3.0,null,null,38.0,1.0,0.0,null,31.0,null,null
"""max""",891.0,1.0,3.0,"""van Melkebeke, Mr. Philemon""","""male""",80.0,8.0,6.0,"""WE/P 5735""",512.3292,"""T""","""S"""


## 3. Количество пассажиров каждого класса

In [3]:
# get_column() достаёт столбец как Series, value_counts() считает повторения
passengers_by_class = (
    titanic.get_column("Pclass")
    .value_counts()
    .sort("Pclass")
    .rename({"count": "пассажиров"})
    .with_columns((pl.col("пассажиров") / titanic.height * 100).round(2).alias("доля, %"))
)
passengers_by_class

Pclass,пассажиров,"доля, %"
i64,u64,f64
1,216,24.24
2,184,20.65
3,491,55.11


## 4. Количество выживших мужчин и женщин

In [4]:
# Survived хранит 0 и 1, поэтому sum() внутри agg() даёт число выживших в группе,
# а len() — общее число пассажиров этой группы
survived_by_sex = (
    titanic.group_by("Sex")
    .agg(
        pl.col("Survived").sum().alias("выжило"),
        pl.col("Survived").len().alias("всего"),
    )
    .with_columns((pl.col("выжило") / pl.col("всего") * 100).round(2).alias("процент выживших"))
    .sort("Sex")
)
survived_by_sex

Sex,выжило,всего,процент выживших
str,i64,u64,f64
"""female""",233,314,74.2
"""male""",109,577,18.89


## 5. Пассажиры старше 44 лет

In [5]:
# filter() оставляет строки, для которых выражение истинно.
# У пассажиров с неизвестным возрастом в Age стоит null: сравнение null > 44
# даёт null, а не True, поэтому в выборку они не попадают
older_44 = titanic.filter(pl.col("Age") > 44)

print(f"пассажиров старше 44 лет: {older_44.height} из {titanic.height}")
older_44.select(["PassengerId", "Name", "Sex", "Age", "Pclass", "Survived"]).head(10)

пассажиров старше 44 лет: 115 из 891


PassengerId,Name,Sex,Age,Pclass,Survived
i64,str,str,f64,i64,i64
7,"""McCarthy, Mr. Timothy J""","""male""",54.0,1,0
12,"""Bonnell, Miss. Elizabeth""","""female""",58.0,1,1
16,"""Hewlett, Mrs. (Mary D Kingcome…","""female""",55.0,2,1
34,"""Wheadon, Mr. Edward H""","""male""",66.0,2,0
53,"""Harper, Mrs. Henry Sleeper (My…","""female""",49.0,1,1
55,"""Ostby, Mr. Engelhart Cornelius""","""male""",65.0,1,0
63,"""Harris, Mr. Henry Birkhardt""","""male""",45.0,1,0
93,"""Chaffee, Mr. Herbert Fuller""","""male""",46.0,1,0
95,"""Coxon, Mr. Daniel""","""male""",59.0,3,0


# Ускорение работы с pandas

## 6. Чтение датасета pandas: средний возраст и стандартное отклонение через bottleneck

In [6]:
import bottleneck as bn
import numpy as np
import pandas as pd

df = pd.read_csv(TITANIC, index_col="PassengerId")

# bottleneck работает с массивами numpy, а не с Series pandas
age = df["Age"].to_numpy()

mean_age = bn.nanmean(age)         # nan-версия игнорирует пропуски, как pandas
std_age = bn.nanstd(age, ddof=1)   # ddof=1 — выборочное отклонение, как у .std() в pandas

print(f"средний возраст:        {mean_age:.4f}")
print(f"стандартное отклонение: {std_age:.4f}")
print(f"значений учтено:        {np.count_nonzero(~np.isnan(age))} из {len(age)}")

# Для контроля: у nanstd по умолчанию ddof=0 (отклонение по генеральной
# совокупности), и результат отличается от pandas менее чем на процент
print("\n--- для сравнения ---")
print(f"pandas  .mean() = {df['Age'].mean():.4f}   .std() = {df['Age'].std():.4f}")
print(f"bn.nanstd(age) без ddof = {bn.nanstd(age):.4f}  <- ddof=0 по умолчанию")
print(f"bn.nanstd совпадает с pandas .std(): {np.isclose(std_age, df['Age'].std())}")

средний возраст:        29.6991
стандартное отклонение: 14.5265
значений учтено:        714 из 891

--- для сравнения ---
pandas  .mean() = 29.6991   .std() = 14.5265
bn.nanstd(age) без ddof = 14.5163  <- ddof=0 по умолчанию
bn.nanstd совпадает с pandas .std(): True


## 7. Новый столбец Fare_new = Fare × 1.3

In [7]:
def fare_new_itertuples(frame: pd.DataFrame) -> list:
    """Способ 1 из задания: обход строк именованными кортежами, Fare * 1.3."""
    return [row.Fare * 1.3 for row in frame.itertuples()]


def fare_new_apply(frame: pd.DataFrame) -> pd.Series:
    """Способ 2 из задания: apply() применяет функцию к каждому значению столбца."""
    return frame["Fare"].apply(lambda fare: fare * 1.3)


df["Fare_new"] = fare_new_itertuples(df)

# Оба способа из задания обязаны дать одно и то же, а заодно сверим с вектором
assert np.allclose(df["Fare_new"], fare_new_apply(df))
assert np.allclose(df["Fare_new"], df["Fare"] * 1.3)

df[["Fare", "Fare_new"]].head()

,Fare,Fare_new
PassengerId,,
1,7.2500,9.42500
2,71.2833,92.66829
3,7.9250,10.30250
4,53.1000,69.03000
5,8.0500,10.46500


In [8]:
import timeit


def micros_per_call(function, number: int = 20) -> float:
    """Среднее время одного вызова в микросекундах (лучший из трёх замеров)."""
    return min(timeit.repeat(function, number=number, repeat=3)) / number * 1e6


print(f"itertuples():  {micros_per_call(lambda: fare_new_itertuples(df)):7.1f} мкс")
print(f"apply():       {micros_per_call(lambda: fare_new_apply(df)):7.1f} мкс")
print(f"векторно:      {micros_per_call(lambda: df['Fare'] * 1.3):7.1f} мкс")
print("\nоба способа из задания работают построчно, поэтому на 891 строке они "
      "в сотни раз медленнее векторного умножения — но задание просит именно их")

itertuples():   6175.5 мкс
apply():         589.0 мкс
векторно:        212.8 мкс

оба способа из задания работают построчно, поэтому на 891 строке они в сотни раз медленнее векторного умножения — но задание просит именно их


# Оптимизация типов pandas

## 8. Чтение датасета Housing.csv

In [9]:
housing = pd.read_csv(HOUSING)
print(f"строк: {housing.shape[0]}, столбцов: {housing.shape[1]}")
housing.head()

строк: 545, столбцов: 13


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


## 9. Оптимальный тип для каждого столбца

In [10]:
# Что вообще лежит в столбцах: тип pandas, число уникальных значений и границы,
# плюс сколько памяти занимает каждый столбец (deep=True считает и память строк)
overview = pd.DataFrame({
    "тип pandas": housing.dtypes.astype(str),
    "уникальных": housing.nunique(),
    "минимум": [housing[column].min() for column in housing.columns],
    "максимум": [housing[column].max() for column in housing.columns],
    "память, КБ": (housing.memory_usage(deep=True)[1:] / 1024).round(1),
})
overview

,тип pandas,уникальных,минимум,максимум,"память, КБ"
price,int64,219,1750000,13300000,4.3
area,int64,284,1650,16200,4.3
bedrooms,int64,6,1,6,4.3
bathrooms,int64,4,1,4,4.3
stories,int64,4,1,4,4.3
mainroad,str,2,no,yes,27.6
guestroom,str,2,no,yes,27.2
basement,str,2,no,yes,27.3
hotwaterheating,str,2,no,yes,27.2
airconditioning,str,2,no,yes,27.3


In [11]:
# ВЫВОДЫ: какой тип данных оптимален для каждого столбца
#
# Числовые столбцы. pandas по умолчанию берёт int64 — 8 байт на значение, хотя
# реальные диапазоны в разы меньше. Подбираем наименьший целый тип, который
# вмещает максимум столбца, иначе будет переполнение:
#
#   price               1 750 000 … 13 300 000  ->  int32  (вмещает до 2 147 483 647)
#   area                    1 650 … 16 200      ->  int16  (вмещает до 32 767)
#   bedrooms                    1 … 6           ->  int8   (вмещает до 127)
#   bathrooms                   1 … 4           ->  int8
#   stories                     1 … 4           ->  int8
#   parking                     0 … 3           ->  int8
#
# Столбцы «да/нет». Шесть столбцов принимают только yes и no — это булев признак.
# pandas хранит каждый такой текст отдельным объектом Python (десятки байт на
# значение), а bool — ровно 1 байт, то есть экономия здесь самая большая:
#
#   mainroad, guestroom, basement,
#   hotwaterheating, airconditioning, prefarea  ->  bool
#
# furnishingstatus принимает три значения (furnished, semi-furnished, unfurnished) —
# булевым его не сделать, но и держать текстом незачем: category хранит код в
# 1 байт плюс один общий словарь категорий на весь столбец:
#
#   furnishingstatus  ->  category
#
# Итог: int64 не нужен ни одному столбцу, а текстовые значения не нужны вовсе.

OPTIMAL_INT_TYPES = {
    "price": "int32",
    "area": "int16",
    "bedrooms": "int8",
    "bathrooms": "int8",
    "stories": "int8",
    "parking": "int8",
}
YES_NO_COLUMNS = ["mainroad", "guestroom", "basement",
                  "hotwaterheating", "airconditioning", "prefarea"]

print("выбранные типы:")
for column, dtype in OPTIMAL_INT_TYPES.items():
    print(f"  {column:<18} {str(housing[column].dtype):<8} -> {dtype}")
for column in YES_NO_COLUMNS:
    print(f"  {column:<18} {str(housing[column].dtype):<8} -> bool")
print(f"  {'furnishingstatus':<18} {str(housing['furnishingstatus'].dtype):<8} -> category")

выбранные типы:
  price              int64    -> int32
  area               int64    -> int16
  bedrooms           int64    -> int8
  bathrooms          int64    -> int8
  stories            int64    -> int8
  parking            int64    -> int8
  mainroad           str      -> bool
  guestroom          str      -> bool
  basement           str      -> bool
  hotwaterheating    str      -> bool
  airconditioning    str      -> bool
  prefarea           str      -> bool
  furnishingstatus   str      -> category


## 10. Смена типов и сравнение потребления памяти

In [12]:
before = housing.memory_usage(deep=True).sum()

# 1) числовые столбцы — в минимально подходящий целый тип
optimized = housing.astype(OPTIMAL_INT_TYPES)
# 2) «да/нет» — в булев тип: eq("yes") даёт True, где было "yes"
optimized[YES_NO_COLUMNS] = optimized[YES_NO_COLUMNS].eq("yes")
# 3) три категории мебели — в category
optimized["furnishingstatus"] = optimized["furnishingstatus"].astype("category")

after = optimized.memory_usage(deep=True).sum()

# Значения обязаны остаться прежними — проверяем каждый затронутый столбец
assert (optimized["price"] == housing["price"]).all()
assert (optimized["area"] == housing["area"]).all()
assert (optimized["mainroad"] == (housing["mainroad"] == "yes")).all()
assert (optimized["furnishingstatus"].astype(str) == housing["furnishingstatus"]).all()

comparison = pd.DataFrame({
    "тип до": housing.dtypes.astype(str),
    "тип после": optimized.dtypes.astype(str),
    "байт до": housing.memory_usage(deep=True)[1:],
    "байт после": optimized.memory_usage(deep=True)[1:],
})
comparison["экономия, %"] = ((1 - comparison["байт после"] / comparison["байт до"]) * 100).round(1)
comparison

,тип до,тип после,байт до,байт после,"экономия, %"
price,int64,int32,4360,2180,50.0
area,int64,int16,4360,1090,75.0
bedrooms,int64,int8,4360,545,87.5
bathrooms,int64,int8,4360,545,87.5
stories,int64,int8,4360,545,87.5
mainroad,str,bool,28263,545,98.1
guestroom,str,bool,27892,545,98.0
basement,str,bool,27986,545,98.1
hotwaterheating,str,bool,27820,545,98.0
airconditioning,str,bool,27967,545,98.1


In [13]:
print(f"до оптимизации:    {before / 1024:7.1f} КБ")
print(f"после оптимизации: {after / 1024:7.1f} КБ")
print(f"экономия:          {(before - after) / 1024:7.1f} КБ, "
      f"{(1 - after / before) * 100:.1f} % (датафрейм легче в {before / after:.1f} раза)")

до оптимизации:      221.9 КБ
после оптимизации:     9.4 КБ
экономия:            212.6 КБ, 95.8 % (датафрейм легче в 23.7 раза)


### Что дало экономию

Основной вклад — текстовые столбцы «да/нет»: каждый `yes` и `no` хранился отдельным объектом Python,
а после перевода в `bool` занимает один байт. Числовые столбцы дали меньший, но бесплатный выигрыш:
`int64` избыточен для числа спален или этажей. Столбец `furnishingstatus` в `category` хранится
как код плюс словарь из трёх значений на весь столбец.

Данные при этом не изменились: проверки `assert` в предыдущей ячейке сравнивают столбцы до и после
преобразования. Точные числа памяти зависят от версии pandas (в старых версиях текстовые столбцы
имеют тип `object`, в новых — собственный строковый тип), поэтому выводятся по месту, а не вписаны
в текст.